In [1]:
import os 
import wandb
import torch

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from hydra import compose, initialize
from pytorch_lightning.callbacks import ModelCheckpoint

from codefiles.helpers import set_all_seeds, build_model, build_lightningmodule, build_datamodule

os.environ["WANDB_SILENT"] = "true"
torch.set_float32_matmul_precision("high")
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

def main(cfg) -> None:
    wandb.finish()
    set_all_seeds(seed=cfg.seed)
    wandb.init(
        project=cfg.wandb.project,
        group=None if cfg.wandb.group == "None" else cfg.wandb.group,
        config={key: value for key, value in cfg.items()},
    )
    
    checkpointaddon = ""
    if "corrupted_data_protocol" in cfg.modelname:
        if cfg.modelname.corrupted_data_protocol:
            checkpointaddon = "_corrupted"
        else:
            checkpointaddon = "_clean"

    checkpoint_name = f"debugmodel{checkpointaddon}"
    if os.path.exists(f'/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints/{checkpoint_name}.ckpt'):
        os.remove(f'/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints/{checkpoint_name}.ckpt')
    
    checkpoint_callback = ModelCheckpoint(
        monitor=cfg.encoders.monitor.metric, mode=cfg.encoders.monitor.mode,
        dirpath=f"/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints",
        filename=checkpoint_name,
        save_top_k=1,
    )

    model = build_model(cfg)
    lightningmodule = build_lightningmodule(cfg, model)
    datamodule = build_datamodule(cfg)

    trainer = pl.Trainer(
        logger=WandbLogger(project=cfg.wandb.project, dir="wandb/"),
        log_every_n_steps=1,
        accelerator='gpu',
        devices=1,
        max_epochs=cfg.max_epochs,
        precision=cfg.precision,
        #enable_checkpointing=True,
        #callbacks=[checkpoint_callback] 
    )

    trainer.fit(lightningmodule, datamodule)
    trainer.test(ckpt_path='best', datamodule=datamodule)

    wandb.finish()

if __name__ == "__main__":
    CONFIG_NAME = "config"
    with initialize(version_base="1.1", config_path="config"):
        cfg = compose(config_name=f"{CONFIG_NAME}")
    main(cfg)

Global seed set to 420
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

Aborted!
Traceback (most recent call last):
  File "/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/wandb/sdk/wandb_init.py", line 1140, in init
    wi.setup(kwargs)
  File "/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/wandb/sdk/wandb_init.py", line 288, in setup
    wandb_login._login(
  File "/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/wandb/sdk/wandb_login.py", line 298, in _login
    wlogin.prompt_api_key()
  File "/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/wandb/sdk/wandb_login.py", line 221, in prompt_api_key
    key, status = self._prompt_api_key()
  File "/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/wandb/sdk/wandb_login.py", line 201, in _prompt_api_key
    key = apikey.prompt_api_key(
  File "/sc-projects

Exception: problem